# 01 — Unified Route Tokenization for TB2 and Kilter

## What is tokenization and why does it matter?

In natural language processing, **tokenization** is the process of converting raw text into a sequence of discrete symbols (tokens) that a model can process. For example, the sentence "I climb rocks" might be tokenized as `["I", " climb", " rocks"]` using a subword tokenizer like BPE.

For climbing board routes, we face an analogous problem: how do we convert a climb — which is fundamentally a *set of holds at specific positions with specific roles* — into a sequence of tokens that a transformer can learn from?

### Key design decisions in this notebook

1. **Board namespacing**: Each hold token includes the board prefix (e.g., `TB2_p344_start` vs `KILTER_p1084_start`). This prevents placement ID collisions between boards — placement 344 on TB2 is a completely different physical hold than placement 344 on Kilter (in fact, the latter does not exist).

2. **Semantic role mapping**: Different boards use different role IDs (TB2 uses 5/6/7/8, Kilter uses 12/13/14/15), but they all map to the same semantic roles: `start`, `middle`, `finish`, `foot`. This shared vocabulary lets the model learn transferable patterns.

3. **Canonical ordering**: Holds within a route are sorted by (role priority, y-position, x-position). This gives the model a consistent input ordering, similar to how LLMs expect text in left-to-right order.

4. **Special tokens**: Like BERT and GPT, we use special tokens:
   - `<BOS>` (beginning of sequence) — marks the start, like `[CLS]` in BERT
   - `<EOS>` (end of sequence) — marks the end, like `[SEP]` or the end-of-text token in GPT
   - `<PAD>` — for batching sequences of different lengths
   - `<UNK>` — for unknown tokens (safety net)
   - `<CLS>` — used by the grade predictor to pool sequence information
   - `<MASK>` — reserved for future masked language modeling experiments

5. **Conditioning tokens**: Routes are prefixed with board, angle, and grade tokens. This is analogous to how modern LLMs use system prompts or task prefixes to condition generation.

### The analogy to NLP

| NLP Concept | Climbing Board Analog |
|---|---|
| Word / Subword | Hold token (placement + role) |
| Sentence | Route (sequence of holds) |
| Document language | Board type (TB2 vs Kilter) |
| Sentence length | Number of holds in route |
| POS tag | Semantic role (start/middle/finish/foot) |
| Genre / Domain | Angle + Grade conditioning |

This notebook tokenizes climbing routes from **both** supported boards:

- Tension Board 2 Mirror
- Kilter Board Original

The board-specific details are stored in `configs/tb2.json` and `configs/kilter.json`.
The shared tokenization code lives in `src/climbingboardgpt/`.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd

# Set up the project root so we can import our custom package
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

# Import our custom modules
from climbingboardgpt.config import load_board_configs
from climbingboardgpt.data import load_multi_board_data
from climbingboardgpt.tokenization import (
    build_route_records,
    build_token_metadata,
    build_vocab,
    encode,
    make_placement_lookup,
    vocab_payload,
)
from climbingboardgpt.utils import assign_group_splits, write_json, json_safe

## Load board configurations

Each board has its own configuration file (`configs/tb2.json`, `configs/kilter.json`) that specifies:

- **`layout_id`**: Which board layout to use (TB2 Mirror = 10, Kilter Original = 1)
- **`role_definitions`**: Maps semantic role names to board-specific role IDs
  - TB2: start=5, middle=6, finish=7, foot=8
  - Kilter: start=12, middle=13, finish=14, foot=15
- **`max_angle`**: We filter out climbs at extreme angles (>50° for TB2, >55° for Kilter) because those grades are biased toward elite climbers
- **`token_prefix`**: The namespace prefix for hold tokens ("TB2" vs "KILTER")
- **`include_mirror_placement_id`**: Whether to include mirror information (TB2 has symmetric left/right holds)

This configuration-driven approach means we can add new boards by creating a new JSON config file, without changing any code.

In [ ]:
configs = load_board_configs(["tb2", "kilter"])
configs

## Load raw climbs and placement metadata

The data loading step reads from SQLite databases downloaded using BoardLib:

```bash
boardlib database tension data/raw/tb2.db
boardlib database kilter data/raw/kilter.db
```

### What we're loading

**Climbs data** (`df_climbs`): Each row is a climb-angle observation. Key columns:
- `uuid`: Unique climb identifier
- `frames`: The raw string encoding holds and roles, e.g., `p344r5p369r6p603r7`
- `angle`: Wall angle in degrees
- `display_difficulty`: Numeric difficulty score (maps to V-grades)
- `boulder_grade`: Human-readable grade like "6b/V4"

**Placements data** (`df_placements`): Each row is a physical hold position on the board. Key columns:
- `placement_id`: The hold's unique ID within its board
- `x`, `y`: Physical coordinates on the board (in inches)
- `default_role_id`: What role this hold typically plays (hand vs foot)
- `set_name`: Material type ("Wood" or "Plastic")
- `mirror_placement_id`: For TB2, the ID of the symmetric hold on the other side

In [ ]:
df_climbs, df_placements = load_multi_board_data(configs, project_root=ROOT)
print(f"Total climbs loaded: {len(df_climbs):,}")
print(f"Total placements loaded: {len(df_placements):,}")
print()
print("Climbs per board:")
print(df_climbs.groupby("board_key").size())

## Build unified route records

This is the core tokenization step. For each climb, we:

1. **Parse the frames string**: Convert `p344r5p369r6p603r7` into a list of `(placement_id, role_id)` tuples

2. **Map role IDs to semantic roles**: Convert board-specific role IDs (5→start, 6→middle, etc.) to shared semantic names

3. **Canonicalize hold order**: Sort holds by (role priority, y-position, x-position). This is important because:
   - The same climb can be represented with holds in any order in the database
   - Transformers need consistent input ordering to learn patterns
   - This is analogous to how NLP tokenizers normalize text (lowercasing, etc.)

4. **Generate token sequences**: Create two versions of each route:
   - `sequence_with_grade`: `<BOS> <BOARD_TB2> <ANGLE_40> <GRADE_V6> <TB2_p344_start> ... <EOS>`
   - `sequence_no_grade`: `<BOS> <BOARD_TB2> <ANGLE_40> <TB2_p344_start> ... <EOS>` (grade removed)

The grade-included version is used for the GPT generator (which predicts the next token, including grade). The grade-excluded version is used for the grade predictor (which receives the route without knowing the grade and must predict it).

In [ ]:
configs_by_key = {config.board_key: config for config in configs}
configs_by_prefix = {config.token_prefix: config for config in configs}
placement_lookup = make_placement_lookup(df_placements)

df_routes = build_route_records(
    df_climbs=df_climbs,
    configs_by_key=configs_by_key,
    placement_lookup=placement_lookup,
)
print(f"Tokenized routes: {len(df_routes):,}")
print()
df_routes[["board_key", "angle", "display_difficulty", "sequence_with_grade"]].head()

## Example tokenized routes

Let's look at what the tokenized routes actually look like. This is the "text" that our transformer models will read.

In [ ]:
for _, row in df_routes.groupby("board_key").head(2).iterrows():
    print(f"Board: {row['board_key']}")
    print(f"  Angle: {row['angle']}°")
    print(f"  Grade: {row['boulder_grade']} (V{row['grouped_v']})")
    print(f"  Tokens: {row['sequence_with_grade']}")
    print()

## Build the shared vocabulary

### What is a vocabulary?

In NLP, the **vocabulary** (or "vocab") is the set of all possible tokens the model can produce or understand. For GPT-3, this is ~50,000 BPE tokens. For our climbing model, it includes:

1. **Special tokens** (6): `<PAD>`, `<UNK>`, `<BOS>`, `<EOS>`, `<CLS>`, `<MASK>`
2. **Board tokens** (2): `<BOARD_TB2>`, `<BOARD_KILTER>`
3. **Angle tokens** (~6): `<ANGLE_30>`, `<ANGLE_35>`, `<ANGLE_40>`, etc.
4. **Grade tokens** (~17): `<GRADE_V0>` through `<GRADE_V16>`
5. **Hold tokens** (~1000+): One per placement per board per role

### Why board-namespaced hold tokens?

Placement ID 344 on TB2 refers to a completely different physical hold than placement ID 344 on Kilter. By prefixing with the board name (`TB2_p344_start` vs `KILTER_p344_start`), we ensure the model treats these as distinct tokens.

This is analogous to how multilingual LLMs use language-specific subword tokens — the same byte sequence can mean different things in different languages.

### String-to-integer mapping

Transformers operate on integer indices, not strings. The `stoi` (string-to-integer) and `itos` (integer-to-string) dictionaries provide this mapping, similar to how HuggingFace tokenizers work.

In [ ]:
vocab_tokens, stoi, itos = build_vocab(df_routes)

print(f"Vocabulary size: {len(stoi):,}")
print()
print("First 20 tokens (special + board tokens):")
print(vocab_tokens[:20])
print()
hold_tokens = [t for t in vocab_tokens if t.startswith('<') and '_p' in t]
angle_tokens = [t for t in vocab_tokens if t.startswith('<ANGLE_')]
grade_tokens = [t for t in vocab_tokens if t.startswith('<GRADE_')]
board_tokens = [t for t in vocab_tokens if t.startswith('<BOARD_')]
special_tokens = [t for t in vocab_tokens if t in ['<PAD>', '<UNK>', '<BOS>', '<EOS>', '<CLS>', '<MASK>']]

print(f"Special tokens: {len(special_tokens)}")
print(f"Board tokens: {len(board_tokens)}")
print(f"Angle tokens: {len(angle_tokens)}")
print(f"Grade tokens: {len(grade_tokens)}")
print(f"Hold tokens: {len(hold_tokens)}")

## Train/validation/test splits

### Why stratified splitting?

We split data into train (80%), validation (10%), and test (10%) sets. Crucially, we **stratify by `board_key × grouped_v`** — this ensures that:

1. Both boards (TB2 and Kilter) are represented in all splits
2. All difficulty levels (V0 through V16) are represented in all splits

Without stratification, we might end up with all V14 climbs in the test set and none in training, which would make evaluation meaningless.

This is the same principle as stratified splitting in NLP, where you ensure all languages or domains are represented in each split.

In [ ]:
df_routes["ids_with_grade"] = df_routes["tokens_with_grade"].apply(lambda tokens: encode(tokens, stoi))
df_routes["ids_no_grade"] = df_routes["tokens_no_grade"].apply(lambda tokens: encode(tokens, stoi))
df_routes["split_stratum"] = df_routes["board_key"].astype(str) + "__V" + df_routes["grouped_v"].astype(str)
df_routes["split"] = assign_group_splits(
    df_routes,
    group_cols=["board_key", "uuid"],
    test_size=0.20,
    val_size_within_temp=0.50,
    random_state=3,
    stratify_col="split_stratum",
)

df_routes.groupby(["board_key", "split"]).size().unstack(fill_value=0)

## Token metadata

### Why metadata matters

Each hold token carries rich metadata that the model can use:

- **Physical coordinates** (`x`, `y`): Where the hold is on the board
- **Normalized coordinates** (`x_norm`, `y_norm`): Scaled to [-1, 1] per board, so the model doesn't need to learn different coordinate scales
- **Semantic role** (`start`, `middle`, `finish`, `foot`): What the hold is used for
- **Board identity** (`board_key`): Which board this hold belongs to

The grade predictor uses these coordinate features as additional embeddings alongside the token embeddings. This is similar to how some LLMs inject positional embeddings or segment embeddings — we're giving the model extra structured information about each token.

In [ ]:
df_token_meta = build_token_metadata(
    vocab_tokens=vocab_tokens,
    stoi=stoi,
    df_placements=df_placements,
    placement_lookup=placement_lookup,
    configs_by_prefix=configs_by_prefix,
)

print("Token metadata columns:")
print(df_token_meta.columns.tolist())
print()
print("Example hold token metadata:")
df_token_meta[df_token_meta["kind"] == "hold"].head()

## Save artifacts

We save several files that will be consumed by later notebooks:

1. **`route_sequences.csv`**: The main tokenized dataset with train/val/test splits
2. **`routes_tokenized.jsonl`**: Same data in JSON Lines format (one JSON object per route)
3. **`token_vocab.json`**: The vocabulary mapping (stoi and itos)
4. **`token_metadata.csv`**: Metadata for each token (coordinates, roles, etc.)
5. **`placement_metadata.csv`**: Physical placement information
6. **`board_summary.csv`**: Aggregate statistics per board

In [ ]:
OUT = ROOT / "data" / "processed" / "tokenized"
OUT.mkdir(parents=True, exist_ok=True)

csv_cols = [
    "uuid", "board_key", "board_display_name", "board_token_prefix", "board_token",
    "climb_name", "setter_username", "layout_id", "layout_name", "board_name",
    "frames", "angle", "display_difficulty", "grouped_v", "boulder_grade",
    "ascensionist_count", "quality_average", "fa_at",
    "n_holds", "n_start", "n_middle", "n_foot", "n_finish",
    "sequence_with_grade", "sequence_no_grade", "split",
]
df_routes[csv_cols].to_csv(OUT / "route_sequences.csv", index=False)

df_placements.to_csv(OUT / "placement_metadata.csv", index=False)

df_token_meta.to_csv(OUT / "token_metadata.csv", index=False)

write_json(OUT / "token_vocab.json", vocab_payload(stoi, itos, configs_by_key))

with (OUT / "routes_tokenized.jsonl").open("w", encoding="utf-8") as handle:
    for record in df_routes.to_dict(orient="records"):
        handle.write(json.dumps(json_safe(record)) + "\n")

board_summary = (
    df_routes.groupby("board_key")
    .agg(
        n_routes=("uuid", "count"),
        mean_angle=("angle", "mean"),
        mean_display_difficulty=("display_difficulty", "mean"),
        mean_holds=("n_holds", "mean"),
    )
    .reset_index()
)
board_summary.to_csv(OUT / "board_summary.csv", index=False)

print("Saved artifacts to:", OUT)
print(f"  - route_sequences.csv ({len(df_routes):,} rows)")
print(f"  - routes_tokenized.jsonl")
print(f"  - token_vocab.json ({len(stoi):,} tokens)")
print(f"  - token_metadata.csv ({len(df_token_meta):,} rows)")
print(f"  - placement_metadata.csv ({len(df_placements):,} rows)")
print(f"  - board_summary.csv")